In [33]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

In [34]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

fasta_file = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins.fasta"
)

records = list(SeqIO.parse(fasta_file, "fasta"))

print(f"first id {records[0].id}")
print(f"first id {records[0].description}")

first id XP_083981534.1
first id XP_083981534.1 ATM [organism=Abramis brama] [GeneID=311090226] [isoform=<NA>]


In [35]:
assert any(record.id == "NP_000042.3" for record in records), "Human ATM reference missing!"
assert min(len(record) for record in records) > 1000, "Suspiciously short sequence found"

In [36]:
import subprocess

aligned_fasta = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins_aligned.fasta"
)

mafft_path = Path(r"C:\Tools\mafft-7.526-win64-signed\mafft-win\mafft.bat")
with open(aligned_fasta, "w") as output_handle:
    alignment_run = subprocess.run(
        [
            "cmd.exe",
            "/C",
            str(mafft_path),
            "--auto",
            "--thread",
            "-1",
            str(fasta_file),
        ],
        stdout=output_handle,
        stderr=subprocess.PIPE,
        text=True,
    )

In [37]:
print("Return Code: ", alignment_run.returncode)
print("Output file exists: ", aligned_fasta.exists())

if aligned_fasta.exists():
    print("Output file size: ", aligned_fasta.stat().st_size, "bytes")

if alignment_run.returncode != 0:
    print(alignment_run.stderr)
    raise RuntimeError("MAFFT alignment failed")

Return Code:  0
Output file exists:  True
Output file size:  3745802 bytes


In [38]:
aligned_records = list(SeqIO.parse(aligned_fasta, "fasta"))

aligned_lengths = {
    len(record.seq)
    for record in aligned_records
}

print("Aligned records:", len(aligned_records))
print("Unique aligned lengths:", aligned_lengths)
print("Human reference present:", any(
    record.id == "NP_000042.3"
    for record in aligned_records
))

Aligned records: 827
Unique aligned lengths: {4301}
Human reference present: True
